In [205]:
import pandas as pd
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("copy_on_write", True)

#### Get the flood dataset

In [206]:
# MERGED TWICE TO REMOVE DUPLICATES

flood_county = pd.read_csv(
    "../../02_processed_data/fema_flood_county_2000.csv"
).reset_index(drop=True)
flood_county.sort_values(by=["FIPS", "YEAR"]).dropna()
print(flood_county.info())
print(flood_county.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35450 entries, 0 to 35449
Data columns (total 7 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   FIPS                   35450 non-null  int64  
 1   YEAR                   35450 non-null  int64  
 2   STATE                  35450 non-null  object 
 3   CZ_NAME                35450 non-null  object 
 4   COUNT                  35450 non-null  int64  
 5   TOTAL_DAMAGE_PROPERTY  35450 non-null  float64
 6   TOTAL_DURATION_HOURS   35450 non-null  float64
dtypes: float64(2), int64(3), object(2)
memory usage: 1.9+ MB
None
               FIPS          YEAR         COUNT  TOTAL_DAMAGE_PROPERTY  \
count  35450.000000  35450.000000  35450.000000           3.545000e+04   
mean   30434.366008   2010.019069     13.039126           3.367555e+06   
std    14880.622613      6.061159     18.316309           1.256051e+08   
min     1001.000000   2000.000000      1.000000           0.0

#### Get the HPI dataset

In [207]:
hpi_county = pd.read_csv("../../02_processed_data/hpi_county_2000.csv")
hpi_county.sort_values(by=["county_fips5", "yr"]).dropna()
print(hpi_county.info())
print(hpi_county.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103294 entries, 0 to 103293
Data columns (total 6 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   state_name                103294 non-null  object 
 1   county/county_equivalent  103294 non-null  object 
 2   county_fips5              103294 non-null  int64  
 3   yr                        103294 non-null  int64  
 4   hpi_change                99020 non-null   float64
 5   index_nsa                 102279 non-null  float64
dtypes: float64(2), int64(2), object(2)
memory usage: 4.7+ MB
None
        county_fips5             yr    hpi_change      index_nsa
count  103294.000000  103294.000000  99020.000000  102279.000000
mean    30137.823339    2004.655120      4.216125     251.727208
std     15165.102342      12.617515      6.537448     194.121414
min      1001.000000    1975.000000    -48.180000      45.290000
25%     18107.000000    1995.000000      0.740000 

#### Checking for missing counties

In [208]:
# hpi_county[hpi_county["cbsa_code"] == "10100"]
flood_county[flood_county["FIPS"] == "46045"]

,FIPS,YEAR,STATE,CZ_NAME,COUNT,TOTAL_DAMAGE_PROPERTY,TOTAL_DURATION_HOURS


In [209]:
min(hpi_county["county_fips5"]), max(hpi_county["county_fips5"]), len(
    hpi_county["county_fips5"].unique()
)

(1001, 56045, 2787)

In [210]:
mask = ~hpi_county["county_fips5"].isin(flood_county["FIPS"])
hpi_no_flood = hpi_county.loc[mask, "county_fips5"].unique()
hpi_no_flood

array([ 2090,  2110,  2122,  2130,  2150,  2170,  6105,  8003,  8021,
        8065, 26009, 26023, 26095, 26141, 26153, 41031, 53059])

In [211]:
mask = ~flood_county["FIPS"].isin(hpi_county["county_fips5"])
flood_no_hpi = flood_county.loc[mask, "FIPS"].unique()
flood_no_hpi

array([ 1011,  1063,  1105,  2005,  2007,  2014,  2015,  2017,  2018,
        2021,  2022,  2023,  2024,  2025,  2026,  2027,  2028,  2101,
        2111,  2121,  2125,  2131,  2135,  2141,  2145,  2151,  2152,
        2155,  2161,  2171,  2181,  2203,  2207,  2208,  2209,  2210,
        2211,  2213,  2214,  2215,  2216,  2217,  2218,  2219,  2221,
        2222,  2223,  2224,  2225,  2226,  2227,  4011,  5013,  5039,
        5073,  5077,  5095,  5099,  5117,  8009,  8017,  8023,  8061,
       12043, 12067, 13003, 13007, 13037, 13053, 13065, 13101, 13125,
       13167, 13183, 13239, 13243, 13259, 13265, 13271, 13283, 13301,
       13307, 13309, 16025, 16033, 17003, 17013, 17069, 17151, 17153,
       20017, 20019, 20025, 20033, 20039, 20047, 20049, 20063, 20065,
       20071, 20073, 20075, 20083, 20089, 20097, 20101, 20105, 20129,
       20135, 20137, 20141, 20147, 20153, 20157, 20179, 20183, 20185,
       20187, 20199, 20201, 20205, 20207, 21025, 21039, 21063, 21129,
       21131, 21135,

#### Merge flood and HPI datasets

In [212]:
hpi_flood = pd.merge(
    hpi_county[["county_fips5", "yr", "hpi_change", "index_nsa"]],
    flood_county,
    left_on=["county_fips5", "yr"],
    right_on=["FIPS", "YEAR"],
    how="left",
    validate="1:1",
)

# Identify columns that came from flood_county (excluding join keys)
flood_cols = [c for c in flood_county.columns if c not in ["FIPS", "YEAR"]]

# Impute those flood variables as 0
hpi_flood[flood_cols] = hpi_flood[flood_cols].fillna(0)

# Drop join-key duplicates from the merged df
hpi_flood = hpi_flood.drop(columns=["FIPS", "YEAR"])

hpi_flood.head()

,county_fips5,yr,hpi_change,index_nsa,STATE,CZ_NAME,COUNT,TOTAL_DAMAGE_PROPERTY,TOTAL_DURATION_HOURS
0,1001,1986,NaN,100.00,0,0,0.0,0.0,0.0
1,1001,1987,-1.86,98.14,0,0,0.0,0.0,0.0
2,1001,1988,2.60,100.68,0,0,0.0,0.0,0.0
3,1001,1989,4.30,105.02,0,0,0.0,0.0,0.0
4,1001,1990,-0.33,104.67,0,0,0.0,0.0,0.0


In [213]:
import numpy as np

hpi_flood = hpi_flood.sort_values(["county_fips5", "yr"])

for col in ["STATE", "CZ_NAME"]:
    hpi_flood[col] = hpi_flood[col].replace([0, "0", "0.0"], np.nan)

# 3. Within each county_fips5, copy non-missing values to all years
hpi_flood[["STATE", "CZ_NAME"]] = (
    hpi_flood.groupby("county_fips5")[["STATE", "CZ_NAME"]]
    .ffill()  # fill forward in time
    .bfill()  # fill backward in time (for early years like 2000–2001)
)

# quick check for your example county
hpi_flood.query("county_fips5 == 48059").head()

,county_fips5,yr,hpi_change,index_nsa,STATE,CZ_NAME,COUNT,TOTAL_DAMAGE_PROPERTY,TOTAL_DURATION_HOURS
84768,48059,1998,NaN,100.00,TEXAS,CALLAHAN,0.0,0.0,0.00
84769,48059,1999,4.45,104.45,TEXAS,CALLAHAN,0.0,0.0,0.00
84770,48059,2000,10.36,115.28,TEXAS,CALLAHAN,0.0,0.0,0.00
84771,48059,2001,-0.07,115.20,TEXAS,CALLAHAN,0.0,0.0,0.00
84772,48059,2002,4.21,120.04,TEXAS,CALLAHAN,2.0,807000.0,14.08


In [214]:
hpi_flood.head()

,county_fips5,yr,hpi_change,index_nsa,STATE,CZ_NAME,COUNT,TOTAL_DAMAGE_PROPERTY,TOTAL_DURATION_HOURS
0,1001,1986,NaN,100.00,ALABAMA,AUTAUGA,0.0,0.0,0.0
1,1001,1987,-1.86,98.14,ALABAMA,AUTAUGA,0.0,0.0,0.0
2,1001,1988,2.60,100.68,ALABAMA,AUTAUGA,0.0,0.0,0.0
3,1001,1989,4.30,105.02,ALABAMA,AUTAUGA,0.0,0.0,0.0
4,1001,1990,-0.33,104.67,ALABAMA,AUTAUGA,0.0,0.0,0.0


In [215]:
flood_cols

['STATE', 'CZ_NAME', 'COUNT', 'TOTAL_DAMAGE_PROPERTY', 'TOTAL_DURATION_HOURS']

In [216]:
# CHECKING FOR DUPLICATES
temp = (
    hpi_flood.value_counts(subset=["county_fips5", "yr"])
    .reset_index()
    .sort_values(by=["county_fips5", "yr"])
)
temp[temp["count"] > 1]

,county_fips5,yr,count


#### Get heat dataset

In [217]:
heat_county = pd.read_csv(
    "../../02_processed_data/county_heat_index_2000_2020_with_HSI.csv"
).reset_index(drop=True)
heat_county

,id,name,state,year,value,rank,mean_1901_2000,heat_index,rel_dev,abs_z_year,abs_pct_year,exceed_90,exceed_95,exceed_100,HSI
0,AL-001,Autauga County,Alabama,2000,93.8,125.0,90.9,0.400821,0.031903,1.316932,0.906059,3.8,0.0,0.0,0.546615
1,AL-001,Autauga County,Alabama,2001,88.9,18.0,90.9,0.041761,-0.022002,0.622524,0.720271,0.0,0.0,0.0,0.295779
2,AL-001,Autauga County,Alabama,2002,90.8,60.0,90.9,0.182546,-0.001100,0.838350,0.815985,0.8,0.0,0.0,0.378973
3,AL-001,Autauga County,Alabama,2003,88.0,3.0,90.9,-0.009982,-0.031903,0.611812,0.704963,0.0,0.0,0.0,0.277611
4,AL-001,Autauga County,Alabama,2004,88.3,11.0,90.9,0.016426,-0.028603,1.005642,0.811312,0.0,0.0,0.0,0.411246
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65242,MD-037,St. Mary&,NaN,2016,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
65243,MD-037,St. Mary&,NaN,2017,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
65244,MD-037,St. Mary&,NaN,2018,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
65245,MD-037,St. Mary&,NaN,2019,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [218]:
#### Get the fIPS codes for states

In [219]:
us_states_fips = pd.read_csv("../../01_original_data/us_states_fips.csv").reset_index(
    drop=True
)
us_states_fips.head()

,state,fips,longname
0,AL,1,Alabama
1,AK,2,Alaska
2,AZ,4,Arizona
3,AR,5,Arkansas
4,CA,6,California


#### Create FIPS codes in heat dataset

In [220]:
def process_fips(heat_county):
    county_split = str(heat_county).split("-")
    state = str(
        us_states_fips[us_states_fips["state"] == county_split[0]]["fips"].values[0]
    ).zfill(2)
    county = county_split[1]
    return state + county


heat_county["FIPS"] = heat_county["id"].apply(lambda x: process_fips(x))
heat_county["FIPS"] = heat_county["FIPS"].astype(int)
heat_county.head()

,id,name,state,year,value,rank,mean_1901_2000,heat_index,rel_dev,abs_z_year,abs_pct_year,exceed_90,exceed_95,exceed_100,HSI,FIPS
0,AL-001,Autauga County,Alabama,2000,93.8,125.0,90.9,0.400821,0.031903,1.316932,0.906059,3.8,0.0,0.0,0.546615,1001
1,AL-001,Autauga County,Alabama,2001,88.9,18.0,90.9,0.041761,-0.022002,0.622524,0.720271,0.0,0.0,0.0,0.295779,1001
2,AL-001,Autauga County,Alabama,2002,90.8,60.0,90.9,0.182546,-0.001100,0.838350,0.815985,0.8,0.0,0.0,0.378973,1001
3,AL-001,Autauga County,Alabama,2003,88.0,3.0,90.9,-0.009982,-0.031903,0.611812,0.704963,0.0,0.0,0.0,0.277611,1001
4,AL-001,Autauga County,Alabama,2004,88.3,11.0,90.9,0.016426,-0.028603,1.005642,0.811312,0.0,0.0,0.0,0.411246,1001


In [221]:
# CHECKING FOR DUPLICATES
temp = (
    heat_county.value_counts(subset=["FIPS", "year"])
    .reset_index()
    .sort_values(by=["FIPS", "year"])
)
temp[temp["count"] > 1]

,FIPS,year,count


#### Merge heat and HPI/flood datasets

In [222]:
hpi_flood_heat = pd.merge(
    hpi_flood,
    heat_county[["FIPS", "year", "value", "heat_index", "HSI"]],
    left_on=["county_fips5", "yr"],
    right_on=["FIPS", "year"],
    how="left",
    validate="1:1",
)

# Identify columns that came from heat_county, excluding join keys
heat_cols = ["FIPS", "year", "value", "heat_index", "HSI"]

# Impute only heat-related variables as 0
hpi_flood_heat[heat_cols] = hpi_flood_heat[heat_cols].fillna(0)

# Drop join-key duplicates
hpi_flood_heat = hpi_flood_heat.drop(columns=["FIPS", "year"])

hpi_flood_heat.head()
hpi_flood_heat

,county_fips5,yr,hpi_change,index_nsa,STATE,CZ_NAME,COUNT,TOTAL_DAMAGE_PROPERTY,TOTAL_DURATION_HOURS,value,heat_index,HSI
0,1001,1986,NaN,100.00,ALABAMA,AUTAUGA,0.0,0.0,0.0,0.0,0.000000,0.000000
1,1001,1987,-1.86,98.14,ALABAMA,AUTAUGA,0.0,0.0,0.0,0.0,0.000000,0.000000
2,1001,1988,2.60,100.68,ALABAMA,AUTAUGA,0.0,0.0,0.0,0.0,0.000000,0.000000
3,1001,1989,4.30,105.02,ALABAMA,AUTAUGA,0.0,0.0,0.0,0.0,0.000000,0.000000
4,1001,1990,-0.33,104.67,ALABAMA,AUTAUGA,0.0,0.0,0.0,0.0,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...
103289,56045,2020,4.45,235.01,WYOMING,WESTON,0.0,0.0,0.0,87.0,0.424372,0.460595
103290,56045,2021,4.97,246.69,WYOMING,WESTON,0.0,0.0,0.0,0.0,0.000000,0.000000
103291,56045,2022,6.15,261.87,WYOMING,WESTON,0.0,0.0,0.0,0.0,0.000000,0.000000
103292,56045,2023,1.32,265.32,WYOMING,WESTON,0.0,0.0,0.0,0.0,0.000000,0.000000


In [223]:
temp = (
    hpi_flood_heat.value_counts(subset=["county_fips5", "yr"])
    .reset_index()
    .sort_values(by=["county_fips5", "yr"])
)
temp[temp["count"] > 1]

,county_fips5,yr,count


#### Merge the drought and wildfire datasets

In [224]:
drought_county = pd.read_csv(
    "../../02_processed_data/NCEI_annual_drought_county_FINAL.csv"
).reset_index(drop=True)
drought_county = drought_county[drought_county["FIPS"] != 0]
drought_county.sort_values(by=["FIPS", "Year"]).dropna()
drought_county = drought_county.drop(["County", "State", "State_abbr"], axis=1)

In [225]:
dupes = drought_county[drought_county.duplicated(subset=["FIPS", "Year"], keep=False)]
dupes = dupes.sort_values(by=["FIPS", "Year"])
drought_county = drought_county.drop_duplicates(subset=["FIPS", "Year"], keep="first")
drought_county.head()

,FIPS,Year,Annual_Mean_Index
0,1001,2000,-2.32
1,1001,2001,1.43
2,1001,2002,-0.76
3,1001,2003,2.39
4,1001,2004,0.00


In [226]:
drought_hpi_flood_heat = pd.merge(
    hpi_flood_heat,
    drought_county[["FIPS", "Year", "Annual_Mean_Index"]],
    left_on=["county_fips5", "yr"],
    right_on=["FIPS", "Year"],
    how="left",
    validate="1:1",
)

drought_hpi_flood_heat = drought_hpi_flood_heat.drop(columns=["FIPS", "Year"])

drought_hpi_flood_heat["Annual_Mean_Index"] = drought_hpi_flood_heat[
    "Annual_Mean_Index"
].fillna(0)

drought_hpi_flood_heat.head()

,county_fips5,yr,hpi_change,index_nsa,STATE,CZ_NAME,COUNT,TOTAL_DAMAGE_PROPERTY,TOTAL_DURATION_HOURS,value,heat_index,HSI,Annual_Mean_Index
0,1001,1986,NaN,100.00,ALABAMA,AUTAUGA,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1001,1987,-1.86,98.14,ALABAMA,AUTAUGA,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1001,1988,2.60,100.68,ALABAMA,AUTAUGA,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1001,1989,4.30,105.02,ALABAMA,AUTAUGA,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1001,1990,-0.33,104.67,ALABAMA,AUTAUGA,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [227]:
# CHECKING FOR DUPLICATES
temp = (
    drought_hpi_flood_heat.value_counts(subset=["county_fips5", "yr"])
    .reset_index()
    .sort_values(by=["county_fips5", "yr"])
)
temp[temp["count"] > 1]

,county_fips5,yr,count


In [228]:
wildfire_county = pd.read_csv(
    "../../02_processed_data/wildfire_data_preprocessed.csv"
).reset_index(drop=True)
wildfire_county.sort_values(by=["FIPS", "Year"]).dropna()
wildfire_county = wildfire_county.drop(["State"], axis=1)
wildfire_county.head()

,FIPS,Year,FIRE_FREQUENCY,TOTAL_FIRE_SIZE
0,1001,2003,43,274.8
1,1001,2004,86,744.5
2,1001,2005,63,234.2
3,1001,2006,62,534.7
4,1001,2007,93,477.3


In [229]:
wildfire_drought_hpi_flood_heat = pd.merge(
    drought_hpi_flood_heat,
    wildfire_county[["FIPS", "Year", "FIRE_FREQUENCY", "TOTAL_FIRE_SIZE"]],
    left_on=["county_fips5", "yr"],
    right_on=["FIPS", "Year"],
    how="left",
    validate="1:1",
)

wildfire_drought_hpi_flood_heat = wildfire_drought_hpi_flood_heat.drop(
    columns=["FIPS", "Year"]
)

wildfire_drought_hpi_flood_heat["FIRE_FREQUENCY"] = wildfire_drought_hpi_flood_heat[
    "FIRE_FREQUENCY"
].fillna(0)
wildfire_drought_hpi_flood_heat["TOTAL_FIRE_SIZE"] = wildfire_drought_hpi_flood_heat[
    "TOTAL_FIRE_SIZE"
].fillna(0)

wildfire_drought_hpi_flood_heat.head()

,county_fips5,yr,hpi_change,index_nsa,STATE,CZ_NAME,COUNT,TOTAL_DAMAGE_PROPERTY,TOTAL_DURATION_HOURS,value,heat_index,HSI,Annual_Mean_Index,FIRE_FREQUENCY,TOTAL_FIRE_SIZE
0,1001,1986,NaN,100.00,ALABAMA,AUTAUGA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1001,1987,-1.86,98.14,ALABAMA,AUTAUGA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1001,1988,2.60,100.68,ALABAMA,AUTAUGA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1001,1989,4.30,105.02,ALABAMA,AUTAUGA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1001,1990,-0.33,104.67,ALABAMA,AUTAUGA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [230]:
# CHECKING FOR DUPLICATES
temp = (
    wildfire_drought_hpi_flood_heat.value_counts(subset=["county_fips5", "yr"])
    .reset_index()
    .sort_values(by=["county_fips5", "yr"])
)
temp[temp["count"] > 1]

,county_fips5,yr,count


#### Merge Drought/Wildfire/HPI/Flood/Heat/Hurricane datasets

In [231]:
storm_summary = pd.read_csv(
    "../../02_processed_data/storm_data_aggregated_county.csv", low_memory=False
)
storm_summary["year"] = storm_summary["year"].astype("int")
storm_summary["FIPS"] = storm_summary["FIPS"].astype("int")

huricane_grouped = storm_summary.groupby(["FIPS", "year"], as_index=False).agg(
    deaths_hurricane=("deaths", "sum"),
    injuries_hurricane=("injuries", "sum"),
    damage_hurricane=("damage", "sum"),
)

huricane_grouped.sort_values(by=["FIPS", "year"]).head()

,FIPS,year,deaths_hurricane,injuries_hurricane,damage_hurricane
0,1003,2002,0.0,0.0,75000
1,1003,2004,0.0,0.0,0
2,1003,2005,0.0,0.0,0
3,1003,2020,2.0,0.0,236080000
4,1013,2004,0.0,0.0,0


In [232]:
wildfire_drought_hpi_flood_heat_huricane = pd.merge(
    wildfire_drought_hpi_flood_heat,
    huricane_grouped,
    left_on=["county_fips5", "yr"],
    right_on=["FIPS", "year"],
    how="left",
    validate="1:1",
)

# Columns coming from huricane_grouped, excluding join keys
hurricane_cols = [c for c in huricane_grouped.columns if c not in ["FIPS", "year"]]

# Impute only hurricane variables as 0
wildfire_drought_hpi_flood_heat_huricane[hurricane_cols] = (
    wildfire_drought_hpi_flood_heat_huricane[hurricane_cols].fillna(0)
)


wildfire_drought_hpi_flood_heat_huricane.head()

,county_fips5,yr,hpi_change,index_nsa,STATE,CZ_NAME,COUNT,TOTAL_DAMAGE_PROPERTY,TOTAL_DURATION_HOURS,value,heat_index,HSI,Annual_Mean_Index,FIRE_FREQUENCY,TOTAL_FIRE_SIZE,FIPS,year,deaths_hurricane,injuries_hurricane,damage_hurricane
0,1001,1986,NaN,100.00,ALABAMA,AUTAUGA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0
1,1001,1987,-1.86,98.14,ALABAMA,AUTAUGA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0
2,1001,1988,2.60,100.68,ALABAMA,AUTAUGA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0
3,1001,1989,4.30,105.02,ALABAMA,AUTAUGA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0
4,1001,1990,-0.33,104.67,ALABAMA,AUTAUGA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,0.0,0.0


In [233]:
climate = wildfire_drought_hpi_flood_heat_huricane

# columns to drop
cols_to_drop = [
    "YEAR",
    "year_y",
    "State_fire",
    "State_ncei",
    "State_abbr",
    "Year",
    "year_x",
    "FIPS",
    "year",
    "value",
    "heat_index",
    "CZ_NAME",
]

# drop them from df (ignore any that might be missing)
climate = climate.drop(columns=cols_to_drop, errors="ignore")

# quick check
print(climate.columns)
print(climate.info())
print(climate.describe())

Index(['county_fips5', 'yr', 'hpi_change', 'index_nsa', 'STATE', 'COUNT',
       'TOTAL_DAMAGE_PROPERTY', 'TOTAL_DURATION_HOURS', 'HSI',
       'Annual_Mean_Index', 'FIRE_FREQUENCY', 'TOTAL_FIRE_SIZE',
       'deaths_hurricane', 'injuries_hurricane', 'damage_hurricane'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103294 entries, 0 to 103293
Data columns (total 15 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   county_fips5           103294 non-null  int64  
 1   yr                     103294 non-null  int64  
 2   hpi_change             99020 non-null   float64
 3   index_nsa              102279 non-null  float64
 4   STATE                  103294 non-null  object 
 5   COUNT                  103294 non-null  float64
 6   TOTAL_DAMAGE_PROPERTY  103294 non-null  float64
 7   TOTAL_DURATION_HOURS   103294 non-null  float64
 8   HSI                    103294 non-null  float64
 9   Annual_

In [234]:
climate = climate.rename(
    columns={
        "COUNT": "FLOOD_FREQUENCY",
        "TOTAL_DAMAGE_PROPERTY": "FLOOD_PROPERTY_DAMAGE",
        "TOTAL_DURATION_HOURS": "FLOOD_DURATION_HOURS",
        "HSI": "HEAT_STRESS_INDEX",
        "Annual_Mean_Index": "DROUGHT_ANNUAL_MEAN_INDEX",
        "TOTAL_FIRE_SIZE": "FIRE_SIZE",
    }
)
climate.columns

Index(['county_fips5', 'yr', 'hpi_change', 'index_nsa', 'STATE',
       'FLOOD_FREQUENCY', 'FLOOD_PROPERTY_DAMAGE', 'FLOOD_DURATION_HOURS',
       'HEAT_STRESS_INDEX', 'DROUGHT_ANNUAL_MEAN_INDEX', 'FIRE_FREQUENCY',
       'FIRE_SIZE', 'deaths_hurricane', 'injuries_hurricane',
       'damage_hurricane'],
      dtype='object')

In [235]:
gdp = pd.read_csv("../../01_original_data/Financial Data/GDP_cleaned.csv")
gdp_small = gdp[["GeoFIPS", "Year", "Real_GDP"]]

In [236]:
years_per_county = (
    gdp_small.groupby("GeoFIPS")["Year"]
    .nunique()
    .reset_index(name="n_years")
    .sort_values("n_years")
)
years_per_county.head()

,GeoFIPS,n_years
0,1001,23
2111,39059,23
2112,39061,23
2113,39063,23
2114,39065,23


In [237]:
counties_per_year = (
    gdp_small.groupby("Year")["GeoFIPS"]
    .nunique()
    .reset_index(name="n_counties")
    .sort_values("Year")
)
counties_per_year.head()

,Year,n_counties
0,2001,3176
1,2002,3176
2,2003,3176
3,2004,3176
4,2005,3176


# dropping 2000 year data is ideal here as 2000 year data is completely missing here.

In [238]:
climate_gdp = pd.merge(
    climate,
    gdp_small,
    left_on=["county_fips5", "yr"],
    right_on=["GeoFIPS", "Year"],
    how="left",
    validate="1:1",
)
climate_gdp = climate_gdp.drop(columns=["GeoFIPS", "Year"])
climate_gdp.head()

,county_fips5,yr,hpi_change,index_nsa,STATE,FLOOD_FREQUENCY,FLOOD_PROPERTY_DAMAGE,FLOOD_DURATION_HOURS,HEAT_STRESS_INDEX,DROUGHT_ANNUAL_MEAN_INDEX,FIRE_FREQUENCY,FIRE_SIZE,deaths_hurricane,injuries_hurricane,damage_hurricane,Real_GDP
0,1001,1986,NaN,100.00,ALABAMA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
1,1001,1987,-1.86,98.14,ALABAMA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
2,1001,1988,2.60,100.68,ALABAMA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
3,1001,1989,4.30,105.02,ALABAMA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
4,1001,1990,-0.33,104.67,ALABAMA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN


In [239]:
unemployment = pd.read_csv(
    "../../01_original_data/Financial Data/Unemployment2023_cleaned.csv"
)
unemployment_small = unemployment[["FIPS_Code", "Year", "Unemployment_Rate"]]

In [240]:
climate_gdp_unemployment = pd.merge(
    climate_gdp,
    unemployment_small,
    left_on=["county_fips5", "yr"],
    right_on=["FIPS_Code", "Year"],
    how="left",
    validate="1:1",
)
climate_gdp_unemployment = climate_gdp_unemployment.drop(columns=["FIPS_Code", "Year"])
climate_gdp_unemployment.head()

,county_fips5,yr,hpi_change,index_nsa,STATE,FLOOD_FREQUENCY,FLOOD_PROPERTY_DAMAGE,FLOOD_DURATION_HOURS,HEAT_STRESS_INDEX,DROUGHT_ANNUAL_MEAN_INDEX,FIRE_FREQUENCY,FIRE_SIZE,deaths_hurricane,injuries_hurricane,damage_hurricane,Real_GDP,Unemployment_Rate
0,1001,1986,NaN,100.00,ALABAMA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
1,1001,1987,-1.86,98.14,ALABAMA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
2,1001,1988,2.60,100.68,ALABAMA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
3,1001,1989,4.30,105.02,ALABAMA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
4,1001,1990,-0.33,104.67,ALABAMA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN


In [241]:
# rows where unemployment is missing
missing_unemp = (
    climate_gdp_unemployment.loc[
        climate_gdp_unemployment["Unemployment_Rate"].isna(), ["county_fips5", "yr"]
    ]
    .drop_duplicates()
    .sort_values(["county_fips5", "yr"])
)

print(missing_unemp)  # nice table of county + year combos

# if you want them in a single "vector" of tuples:
missing_unemp_vec = list(missing_unemp.itertuples(index=False, name=None))
print(missing_unemp_vec)

# or as strings like "county-year"
missing_unemp_str = [f"{c}-{y}" for c, y in missing_unemp_vec]
print(missing_unemp_str)

        county_fips5    yr
0               1001  1986
1               1001  1987
2               1001  1988
3               1001  1989
4               1001  1990
...              ...   ...
103265         56045  1996
103266         56045  1997
103267         56045  1998
103268         56045  1999
103293         56045  2024

[37706 rows x 2 columns]
[(1001, 1986), (1001, 1987), (1001, 1988), (1001, 1989), (1001, 1990), (1001, 1991), (1001, 1992), (1001, 1993), (1001, 1994), (1001, 1995), (1001, 1996), (1001, 1997), (1001, 1998), (1001, 1999), (1001, 2024), (1003, 1977), (1003, 1978), (1003, 1979), (1003, 1980), (1003, 1981), (1003, 1982), (1003, 1983), (1003, 1984), (1003, 1985), (1003, 1986), (1003, 1987), (1003, 1988), (1003, 1989), (1003, 1990), (1003, 1991), (1003, 1992), (1003, 1993), (1003, 1994), (1003, 1995), (1003, 1996), (1003, 1997), (1003, 1998), (1003, 1999), (1003, 2024), (1005, 1985), (1005, 1986), (1005, 1987), (1005, 1988), (1005, 1989), (1005, 1990), (1005, 1991), (1005

In [242]:
df = climate_gdp_unemployment

all_missing_prop = (
    df.groupby("county_fips5")["Unemployment_Rate"]
    .apply(lambda s: s.isna().all())
    .reset_index(name="all_missing_Unemployment_Rate")
)

n_counties_all_Unemployement_Rate = all_missing_prop[
    "all_missing_Unemployment_Rate"
].sum()
print(
    "Counties with Unemployment rate missing for ALL years:",
    n_counties_all_Unemployement_Rate,
)

# optionally list them
counties_all_missing_prop = all_missing_prop.loc[
    all_missing_prop["all_missing_Unemployment_Rate"], "county_fips5"
].tolist()
print(
    "county_fips5 with Unemployment_Rate missing for all years:",
    n_counties_all_Unemployement_Rate,
)
n_counties_all_Unemployement_Rate

Counties with Unemployment rate missing for ALL years: 0
county_fips5 with Unemployment_Rate missing for all years: 0


np.int64(0)

In [243]:
# total NA count per column
climate_gdp_unemployment.isna().sum().sort_values(ascending=False)

# percentage of NA per column
(climate_gdp_unemployment.isna().mean().sort_values(ascending=False) * 100).round(2)

Real_GDP                     39.96
Unemployment_Rate            36.50
hpi_change                    4.14
index_nsa                     0.98
FIRE_FREQUENCY                0.00
damage_hurricane              0.00
injuries_hurricane            0.00
deaths_hurricane              0.00
FIRE_SIZE                     0.00
county_fips5                  0.00
DROUGHT_ANNUAL_MEAN_INDEX     0.00
yr                            0.00
FLOOD_DURATION_HOURS          0.00
FLOOD_PROPERTY_DAMAGE         0.00
FLOOD_FREQUENCY               0.00
STATE                         0.00
HEAT_STRESS_INDEX             0.00
dtype: float64

In [244]:
property_tax = pd.read_csv(
    "../../01_original_data/Financial Data/property_tax_cleaned.csv"
)

In [245]:
climate_gdp_unemployment_tax = pd.merge(
    climate_gdp_unemployment,
    property_tax,
    left_on=["county_fips5", "yr"],
    right_on=["fips", "year"],
    how="left",
    validate="1:1",
)
climate_gdp_unemployment_tax = climate_gdp_unemployment_tax.drop(
    columns=["fips", "year"]
)
climate_gdp_unemployment_tax.head()

,county_fips5,yr,hpi_change,index_nsa,STATE,FLOOD_FREQUENCY,FLOOD_PROPERTY_DAMAGE,FLOOD_DURATION_HOURS,HEAT_STRESS_INDEX,DROUGHT_ANNUAL_MEAN_INDEX,FIRE_FREQUENCY,FIRE_SIZE,deaths_hurricane,injuries_hurricane,damage_hurricane,Real_GDP,Unemployment_Rate,prop_rate
0,1001,1986,NaN,100.00,ALABAMA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN
1,1001,1987,-1.86,98.14,ALABAMA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN
2,1001,1988,2.60,100.68,ALABAMA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN
3,1001,1989,4.30,105.02,ALABAMA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN
4,1001,1990,-0.33,104.67,ALABAMA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN


### Add Mortgage data

In [246]:
mortgage = pd.read_csv(
    "../../01_original_data/Financial Data/combined_mortgage_agg.csv"
)
mortgage.head()

,YEAR,county_fips,total_home_purchase,total_amt_purchase
0,2000,1001,831,72778000.0
1,2000,1003,2933,301169000.0
2,2000,1005,237,13977000.0
3,2000,1007,272,13339000.0
4,2000,1009,729,55125000.0


In [247]:
climate_gdp_unemployment_tax_mort = climate_gdp_unemployment_tax.merge(
    mortgage,
    "left",
    left_on=["county_fips5", "yr"],
    right_on=["county_fips", "YEAR"],
    validate="1:1",
)
if "YEAR" in climate_gdp_unemployment_tax_mort.columns:
    climate_gdp_unemployment_tax_mort = climate_gdp_unemployment_tax_mort.drop(
        columns=["YEAR"]
    )
climate_gdp_unemployment_tax_mort.head()

,county_fips5,yr,hpi_change,index_nsa,STATE,FLOOD_FREQUENCY,FLOOD_PROPERTY_DAMAGE,FLOOD_DURATION_HOURS,HEAT_STRESS_INDEX,DROUGHT_ANNUAL_MEAN_INDEX,...,FIRE_SIZE,deaths_hurricane,injuries_hurricane,damage_hurricane,Real_GDP,Unemployment_Rate,prop_rate,county_fips,total_home_purchase,total_amt_purchase
0,1001,1986,NaN,100.00,ALABAMA,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,1001,1987,-1.86,98.14,ALABAMA,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,1001,1988,2.60,100.68,ALABAMA,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
3,1001,1989,4.30,105.02,ALABAMA,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
4,1001,1990,-0.33,104.67,ALABAMA,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN


### Add Population data

In [248]:
population = pd.read_csv("../../01_original_data/Financial Data/population_cleaned.csv")
population.head()

,IBRC_Geo_ID,Year,Total Population
0,1000,2000,4452173
1,1000,2001,4467634
2,1000,2002,4480089
3,1000,2003,4503491
4,1000,2004,4530729


In [249]:
# Checking for duplicates
population[population.duplicated(subset=["IBRC_Geo_ID", "Year"], keep=False)]

,IBRC_Geo_ID,Year,Total Population


In [250]:
climate_gdp_unemployment_tax_mort_pop = climate_gdp_unemployment_tax_mort.merge(
    population,
    "left",
    left_on=["county_fips5", "yr"],
    right_on=["IBRC_Geo_ID", "Year"],
    validate="1:1",
)
climate_gdp_unemployment_tax_mort_pop.head()

,county_fips5,yr,hpi_change,index_nsa,STATE,FLOOD_FREQUENCY,FLOOD_PROPERTY_DAMAGE,FLOOD_DURATION_HOURS,HEAT_STRESS_INDEX,DROUGHT_ANNUAL_MEAN_INDEX,...,damage_hurricane,Real_GDP,Unemployment_Rate,prop_rate,county_fips,total_home_purchase,total_amt_purchase,IBRC_Geo_ID,Year,Total Population
0,1001,1986,NaN,100.00,ALABAMA,0.0,0.0,0.0,0.0,0.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1001,1987,-1.86,98.14,ALABAMA,0.0,0.0,0.0,0.0,0.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1001,1988,2.60,100.68,ALABAMA,0.0,0.0,0.0,0.0,0.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1001,1989,4.30,105.02,ALABAMA,0.0,0.0,0.0,0.0,0.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1001,1990,-0.33,104.67,ALABAMA,0.0,0.0,0.0,0.0,0.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Ensuring year 2000 - 2020

In [251]:
# Pick years 2000 - 2020
climate_gdp_unemployment_tax_mort_pop = climate_gdp_unemployment_tax_mort_pop[
    climate_gdp_unemployment_tax_mort_pop["yr"].between(2000, 2020)
]

# Drop redundant columns
if "Year" in climate_gdp_unemployment_tax_mort_pop.columns:
    climate_gdp_unemployment_tax_mort_pop = climate_gdp_unemployment_tax_mort_pop.drop(
        columns=["Year", "IBRC_Geo_ID"]
    )

# Standardize column names
climate_gdp_unemployment_tax_mort_pop.columns = [
    column.lower().replace(" ", "_")
    for column in climate_gdp_unemployment_tax_mort_pop.columns
]

# Fill empties with 0
climate_gdp_unemployment_tax_mort_pop = climate_gdp_unemployment_tax_mort_pop.fillna(0)
climate_gdp_unemployment_tax_mort_pop.head()

,county_fips5,yr,hpi_change,index_nsa,state,flood_frequency,flood_property_damage,flood_duration_hours,heat_stress_index,drought_annual_mean_index,...,deaths_hurricane,injuries_hurricane,damage_hurricane,real_gdp,unemployment_rate,prop_rate,county_fips,total_home_purchase,total_amt_purchase,total_population
14,1001,2000,2.50,141.23,ALABAMA,1.0,200000.0,3.50,0.546615,-2.32,...,0.0,0.0,0.0,0,4.1,0.000,1001.0,831.0,72778000.0,44021.0
15,1001,2001,3.44,146.08,ALABAMA,2.0,37000.0,4.25,0.295779,1.43,...,0.0,0.0,0.0,1041239,4.1,0.000,1001.0,766.0,71592000.0,44889.0
16,1001,2002,2.67,149.98,ALABAMA,0.0,0.0,0.00,0.378973,-0.76,...,0.0,0.0,0.0,1080135,4.8,0.000,1001.0,730.0,78607000.0,45909.0
17,1001,2003,1.79,152.66,ALABAMA,2.0,58000.0,6.00,0.277611,2.39,...,0.0,0.0,0.0,1111465,5.0,0.245,1001.0,835.0,93128000.0,46800.0
18,1001,2004,2.92,157.12,ALABAMA,1.0,8000.0,3.50,0.411246,0.00,...,0.0,0.0,0.0,1283522,4.7,0.245,1001.0,908.0,111256000.0,48366.0


In [252]:
climate_gdp_unemployment_tax_mort_pop.to_csv(
    "../../02_processed_data/FINAL_CLIMATE_FINANCIAL_DATA.csv"
)

In [253]:
climate_gdp_unemployment_tax_mort_pop[climate_gdp_unemployment_tax_mort_pop["index_nsa"] > 1200]

,county_fips5,yr,hpi_change,index_nsa,state,flood_frequency,flood_property_damage,flood_duration_hours,heat_stress_index,drought_annual_mean_index,...,deaths_hurricane,injuries_hurricane,damage_hurricane,real_gdp,unemployment_rate,prop_rate,county_fips,total_home_purchase,total_amt_purchase,total_population
5415,6001,2005,24.08,1471.28,CALIFORNIA,1.0,8800000.0,5.98,0.233633,2.67,...,0.0,0.0,0.0,94611477,5.1,1.14900,6001.0,30836.0,1.198427e+10,1441545.0
5416,6001,2006,8.76,1600.18,CALIFORNIA,1.0,8800000.0,18.00,0.318613,1.44,...,0.0,0.0,0.0,97221046,4.4,1.15900,6001.0,24762.0,9.484389e+09,1444484.0
5417,6001,2007,-3.32,1547.03,CALIFORNIA,0.0,0.0,0.00,0.260261,-2.99,...,0.0,0.0,0.0,98832535,4.7,1.15660,6001.0,15805.0,6.568123e+09,1455715.0
5418,6001,2008,-15.41,1308.66,CALIFORNIA,0.0,0.0,0.00,0.394210,-3.15,...,0.0,0.0,0.0,99236414,6.2,1.16818,6001.0,11475.0,4.255872e+09,1477208.0
5423,6001,2013,13.74,1222.21,CALIFORNIA,0.0,0.0,0.00,0.317704,-2.92,...,0.0,0.0,0.0,106957044,7.3,1.21400,6001.0,12404.0,5.719853e+09,1579508.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
97888,53061,2018,10.15,1304.03,WASHINGTON,0.0,0.0,0.00,0.174326,-0.65,...,0.0,0.0,0.0,51629801,3.6,0.00000,53061.0,12958.0,5.170950e+09,814107.0
97889,53061,2019,4.37,1360.95,WASHINGTON,0.0,0.0,0.00,0.202292,-2.67,...,0.0,0.0,0.0,51878459,2.9,0.00000,53061.0,13327.0,5.562555e+09,822413.0
97890,53061,2020,5.02,1429.26,WASHINGTON,0.0,0.0,0.00,0.119684,0.38,...,0.0,0.0,0.0,45616174,8.8,0.00000,53061.0,14218.0,6.552340e+09,829975.0
98168,53073,2019,7.39,1211.86,WASHINGTON,0.0,0.0,0.00,0.217707,-1.65,...,0.0,0.0,0.0,13672902,5.0,0.00000,53073.0,2897.0,9.954050e+08,228675.0
